In [4]:
"""
Optimized SoundCloud Downloader with Somali ASR Transcription.
Designed for efficient batch processing with minimal redundant metadata.
"""

import os
import re
import hashlib
import json
import time
import shutil
import tempfile
from datetime import datetime, timedelta
from pathlib import Path
from typing import Optional, Dict, Tuple, List
import io

import yt_dlp
import pandas as pd
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import librosa
from tqdm import tqdm

try:
    from pydub import AudioSegment
    PYDUB_AVAILABLE = True
except ImportError:
    PYDUB_AVAILABLE = False


class SomaliASREngine:
    """Transcription engine using Mustafaa4a/ASR-Somali model."""
    
    def __init__(self, model_name: str = "Mustafaa4a/ASR-Somali", verbose: bool = False):
        """
        Initialize Somali ASR model.
        
        Args:
            model_name: HuggingFace model identifier
            verbose: If True, print detailed loading information
        """
        self.processor = None
        self.model = None
        self.device = "cpu"
        self.verbose = verbose
        self._setup_model(model_name)

    def _setup_model(self, model_name: str) -> None:
        """
        Load Wav2Vec2 model and processor.
        
        Raises:
            RuntimeError: If model fails to load
        """
        if self.verbose:
            print(f"Loading ASR model: {model_name}")
            
        try:
            self.processor = Wav2Vec2Processor.from_pretrained(model_name)
            self.model = Wav2Vec2ForCTC.from_pretrained(model_name)
            
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
            self.model.to(self.device)
            
            if self.verbose:
                print(f"✓ Model loaded on {self.device.upper()}")
        except Exception as e:
            raise RuntimeError(f"Failed to load ASR model: {e}")

    def transcribe_from_memory(self, audio_data: bytes) -> str:
        """
        Transcribe audio from memory buffer.

        Args:
            audio_data: Raw audio bytes

        Returns:
            Transcribed text
            
        Raises:
            ValueError: If transcription fails
        """
        target_sr = 16000
        
        try:
            # Load and resample audio
            audio, sr = librosa.load(io.BytesIO(audio_data), sr=target_sr)
            
            # Adaptive chunk sizing based on device
            chunk_length_s = 45 if self.device == 'cuda' else 20
            chunk_length = chunk_length_s * target_sr
            
            transcriptions = []
            
            # Process in chunks without verbose output
            for i in range(0, len(audio), chunk_length):
                chunk = audio[i:i + chunk_length]
                
                input_values = self.processor(
                    chunk,
                    sampling_rate=target_sr,
                    return_tensors="pt",
                    padding=True
                ).input_values.to(self.device)
                
                with torch.no_grad():
                    logits = self.model(input_values).logits
                
                predicted_ids = torch.argmax(logits, dim=-1)
                transcription = self.processor.batch_decode(predicted_ids)[0]
                transcriptions.append(transcription)
            
            return " ".join(transcriptions).strip()

        except Exception as e:
            raise ValueError(f"Transcription failed: {e}")


class StreamingSoundCloudDownloader:
    """
    Batch processor for SoundCloud audio with Somali ASR transcription.
    Optimized for long date ranges with minimal metadata overhead.
    """
    
    def __init__(self, output_dir: str, output_format: str = "structured", verbose: bool = False):
        """
        Initialize downloader.
        
        Args:
            output_dir: Directory for outputs
            output_format: 'structured' (CSV), 'txt', or 'both'
            verbose: Enable detailed logging
        """
        self.output_dir = Path(output_dir)
        self.output_format = output_format
        self.verbose = verbose
        self.ffmpeg_path = self._find_ffmpeg()
        
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        self.log_file = self.output_dir / ".transcription_log.json"
        self.structured_data_file = self.output_dir / "transcriptions_database.csv"
        
        self.transcription_log = self._load_transcription_log()
        self._init_structured_storage()
        
        self.transcription_engine = SomaliASREngine(verbose=verbose)

    def _find_ffmpeg(self) -> Optional[str]:
        """Locate ffmpeg binary."""
        if self.verbose:
            print("Checking for ffmpeg...")
            
        ffmpeg_path = shutil.which('ffmpeg')
        if ffmpeg_path:
            if self.verbose:
                print(f"✓ Found ffmpeg: {ffmpeg_path}")
            return ffmpeg_path
        
        # Check common locations
        for location in ['/usr/bin/ffmpeg', '/home/zeus/miniconda3/bin/ffmpeg']:
            if Path(location).exists():
                if self.verbose:
                    print(f"✓ Found ffmpeg: {location}")
                return location
                
        if self.verbose:
            print("⚠️ ffmpeg not found")
        return None

    def _init_structured_storage(self) -> None:
        """Initialize CSV database with minimal essential columns."""
        if not self.structured_data_file.exists():
            # Optimized schema - removed redundant columns
            columns = [
                'id',
                'url',
                'title',
                'date_recorded',
                'date_processed',
                'processing_duration_seconds',
                'audio_size_mb',
                'audio_duration_seconds',
                'transcript_length_chars',
                'transcript_length_words',
                'transcript_text'
            ]
            pd.DataFrame(columns=columns).to_csv(self.structured_data_file, index=False)

    def _load_transcription_log(self) -> Dict:
        """Load processing log to skip already-processed URLs."""
        if self.log_file.exists():
            try:
                with open(self.log_file, 'r') as f:
                    return json.load(f)
            except json.JSONDecodeError:
                return {}
        return {}

    def _save_transcription_log(self) -> None:
        """Persist processing log."""
        with open(self.log_file, 'w') as f:
            json.dump(self.transcription_log, f, indent=2)

    def _download_to_memory(self, url: str) -> Tuple[Optional[bytes], Dict]:
        """
        Download audio to memory buffer.
        
        Args:
            url: SoundCloud track URL
            
        Returns:
            Tuple of (audio_bytes, metadata_dict)
        """
        metadata = {'success': False, 'error': None, 'info': {}}
        
        ydl_opts = {
            'format': 'bestaudio/best',
            'noplaylist': True,
            'quiet': True,
            'no_warnings': True,
            'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'mp3'}],
        }
        
        if self.ffmpeg_path:
            ydl_opts['ffmpeg_location'] = self.ffmpeg_path

        try:
            # Extract metadata
            with yt_dlp.YoutubeDL({'quiet': True}) as ydl:
                info = ydl.extract_info(url, download=False)
                metadata['info'] = {
                    'title': info.get('title', 'Unknown'),
                    'duration': info.get('duration', 0),
                    'uploader': info.get('uploader', 'Unknown'),
                    'upload_date': info.get('upload_date', ''),
                }
            
            # Download to temporary file
            with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as tmp_file:
                ydl_opts['outtmpl'] = tmp_file.name.replace('.mp3', '.%(ext)s')
                
                with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                    ydl.download([url])
                
                processed_file = tmp_file.name.replace('.mp3', '.mp3')
                
                if Path(processed_file).exists():
                    with open(processed_file, 'rb') as f:
                        audio_data = f.read()
                    
                    Path(processed_file).unlink()
                    metadata['success'] = True
                    return audio_data, metadata
                    
        except Exception as e:
            metadata['error'] = str(e)
            return None, metadata
            
        return None, metadata

    def _transcribe_audio_data(self, audio_data: bytes, metadata: Dict) -> Tuple[Optional[str], Dict]:
        """
        Transcribe audio from memory.
        
        Args:
            audio_data: Raw audio bytes
            metadata: Metadata dictionary to update
            
        Returns:
            Tuple of (transcript_text, updated_metadata)
        """
        start_time = time.time()
        
        if not audio_data:
            metadata['transcription_error'] = "Empty audio data"
            return None, metadata
            
        try:
            metadata['audio_size_mb'] = len(audio_data) / (1024 * 1024)
            
            if PYDUB_AVAILABLE:
                audio_segment = AudioSegment.from_file(io.BytesIO(audio_data))
                metadata['audio_duration_seconds'] = len(audio_segment) / 1000.0

            transcript = self.transcription_engine.transcribe_from_memory(audio_data)
            
            metadata.update({
                'processing_duration_seconds': time.time() - start_time,
                'transcript_length_chars': len(transcript),
                'transcript_length_words': len(transcript.split()),
                'transcription_success': True,
            })
            
            return transcript, metadata
            
        except Exception as e:
            metadata.update({
                'transcription_error': str(e),
                'transcription_success': False,
                'processing_duration_seconds': time.time() - start_time,
            })
            return None, metadata

    def _save_structured_data(self, record: Dict) -> None:
        """Append record to CSV database."""
        try:
            new_row = pd.DataFrame([record])
            new_row.to_csv(self.structured_data_file, mode='a', header=False, index=False)
        except Exception as e:
            if self.verbose:
                print(f"⚠️ Could not save to database: {e}")

    def _save_transcript_file(self, transcript: str, metadata: Dict, base_filename: str) -> str:
        """Save transcript as text file with metadata header."""
        transcript_file = self.output_dir / f"{base_filename}.txt"
        
        with open(transcript_file, 'w', encoding='utf-8') as f:
            f.write(f"# Transcription Report\n{'='*50}\n")
            f.write(f"Source URL: {metadata.get('url', 'Unknown')}\n")
            f.write(f"Title: {metadata.get('info', {}).get('title', 'Unknown')}\n")
            f.write(f"Date Processed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Model: Mustafaa4a/ASR-Somali\n")
            f.write(f"Audio Duration: {metadata.get('audio_duration_seconds', 0):.1f}s\n")
            f.write(f"Processing Time: {metadata.get('processing_duration_seconds', 0):.1f}s\n")
            f.write(f"Word Count: {metadata.get('transcript_length_words', 0)}\n")
            f.write(f"{'='*50}\n\n## Transcript\n\n{transcript}")
            
        return str(transcript_file)

    def _process_url(self, url: str) -> Tuple[bool, Optional[str], Dict]:
        """
        Process single URL: download, transcribe, save.
        
        Args:
            url: SoundCloud track URL
            
        Returns:
            Tuple of (success, file_path, metadata)
        """
        url_hash = hashlib.md5(url.encode()).hexdigest()
        
        # Skip if already processed
        if url_hash in self.transcription_log:
            return True, self.transcription_log[url_hash].get('file_path'), self.transcription_log[url_hash]
        
        # Download
        audio_data, metadata = self._download_to_memory(url)
        if not audio_data:
            return False, None, metadata
            
        metadata['url'] = url
        metadata['id'] = url_hash
        
        # Transcribe
        transcript, metadata = self._transcribe_audio_data(audio_data, metadata)
        if not transcript:
            return False, None, metadata
        
        # Generate safe filename
        safe_title = re.sub(r'[^\w\s-]', '', metadata.get('info', {}).get('title', ''))
        base_filename = re.sub(r'[-\s]+', '-', safe_title).strip('-') or f"soundcloud_{url_hash[:8]}"

        file_path = None
        
        # Save to database with optimized schema
        if self.output_format in ['structured', 'both']:
            record = {
                'id': url_hash,
                'url': url,
                'title': metadata.get('info', {}).get('title', 'Unknown'),
                'date_recorded': metadata.get('info', {}).get('upload_date', ''),
                'date_processed': datetime.now().isoformat(),
                'processing_duration_seconds': metadata.get('processing_duration_seconds', 0),
                'audio_size_mb': metadata.get('audio_size_mb', 0),
                'audio_duration_seconds': metadata.get('audio_duration_seconds', 0),
                'transcript_length_chars': metadata.get('transcript_length_chars', 0),
                'transcript_length_words': metadata.get('transcript_length_words', 0),
                'transcript_text': transcript
            }
            self._save_structured_data(record)
            file_path = str(self.structured_data_file)
        
        # Save text file
        if self.output_format in ['txt', 'both']:
            txt_file = self._save_transcript_file(transcript, metadata, base_filename)
            file_path = txt_file
        
        # Update log
        self.transcription_log[url_hash] = {
            'url': url,
            'title': metadata.get('info', {}).get('title', 'Unknown'),
            'file_path': file_path,
            'success': True
        }
        self._save_transcription_log()
        
        return True, file_path, metadata

    def _generate_urls_for_range(self, profile_url: str, start_date: datetime, end_date: datetime) -> List[Tuple[datetime, str]]:
        """
        Generate SoundCloud URLs for date range.
        
        Args:
            profile_url: Base SoundCloud profile URL
            start_date: Start date (inclusive)
            end_date: End date (inclusive)
            
        Returns:
            List of (date, url) tuples
        """
        urls = []
        profile_url = profile_url.rstrip('/')
        current_date = start_date
        
        while current_date <= end_date:
            day = current_date.day
            month = current_date.strftime('%b').lower()
            year = current_date.year
            
            url = f"{profile_url}/idaacadda-{day:02d}-{month}-{year}"
            urls.append((current_date, url))
            current_date += timedelta(days=1)
            
        return urls

    def process_date_range(self, profile_url: str, start_date_str: str, end_date_str: str) -> Dict:
        """
        Process all tracks for a date range with progress tracking.
        
        Args:
            profile_url: SoundCloud profile URL
            start_date_str: Start date in 'YYYY-MM-DD' format
            end_date_str: End date in 'YYYY-MM-DD' format
            
        Returns:
            Dictionary with 'successful', 'failed', 'skipped' URL lists
            
        Raises:
            ValueError: If date format is invalid
        """
        try:
            start_date = datetime.strptime(start_date_str, '%Y-%m-%d')
            end_date = datetime.strptime(end_date_str, '%Y-%m-%d')
        except ValueError:
            raise ValueError("Invalid date format. Use 'YYYY-MM-DD'")

        print(f"\n{'='*70}")
        print(f"SoundCloud ASR Transcription Pipeline")
        print(f"{'='*70}")
        print(f"Profile: {profile_url}")
        print(f"Date Range: {start_date.date()} to {end_date.date()}")
        print(f"Output: {self.structured_data_file}")
        print(f"{'='*70}\n")
        
        urls = self._generate_urls_for_range(profile_url, start_date, end_date)
        
        if not urls:
            print("No URLs generated for date range")
            return {'successful': [], 'failed': [], 'skipped': []}
        
        results = {'successful': [], 'failed': [], 'skipped': []}
        
        # Process with progress bar
        with tqdm(urls, desc="Processing tracks", unit="track") as pbar:
            for date, url in pbar:
                pbar.set_postfix_str(f"{date.date()}")
                
                try:
                    success, _, _ = self._process_url(url)
                    if success:
                        results['successful'].append(url)
                    else:
                        results['failed'].append(url)
                except Exception as e:
                    if self.verbose:
                        tqdm.write(f"✗ Error on {date.date()}: {e}")
                    results['failed'].append(url)
                
                time.sleep(0.5)  # Rate limiting

        # Summary
        print(f"\n{'='*70}")
        print("Processing Complete")
        print(f"{'='*70}")
        print(f"✓ Successful: {len(results['successful'])}")
        print(f"✗ Failed: {len(results['failed'])}")
        print(f"Database: {self.structured_data_file}")
        
        if results['failed'] and self.verbose:
            print("\nFailed URLs:")
            for url in results['failed']:
                print(f"  - {url}")
        
        print(f"{'='*70}\n")
        
        return results


def stream_transcribe_date_range(
    profile_url: str,
    start_date: str,
    end_date: str,
    output_dir: str,
    output_format: str = "structured",
    verbose: bool = False
) -> Dict:
    """
    High-level function to run date range transcription.
    
    Args:
        profile_url: SoundCloud profile URL
        start_date: Start date 'YYYY-MM-DD'
        end_date: End date 'YYYY-MM-DD'
        output_dir: Output directory path
        output_format: 'structured', 'txt', or 'both'
        verbose: Enable detailed logging
        
    Returns:
        Dictionary with processing results
        
    Example:
        results = stream_transcribe_date_range(
            profile_url="https://soundcloud.com/radio-ergo",
            start_date="2024-07-01",
            end_date="2024-07-31",
            output_dir="./transcripts",
            output_format="structured"
        )
    """
    downloader = StreamingSoundCloudDownloader(
        output_dir=output_dir,
        output_format=output_format,
        verbose=verbose
    )
    return downloader.process_date_range(
        profile_url=profile_url,
        start_date_str=start_date,
        end_date_str=end_date
    )

In [2]:
# Usage example
if __name__ == "__main__":
    OUTPUT_DIR = "/teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/02_intermediate/transcripts/mustafaa4a_ASR-Somali"
    
    results = stream_transcribe_date_range(
        profile_url="https://soundcloud.com/radio-ergo",
        start_date="2024-07-01",
        end_date="2024-07-01",
        output_dir=OUTPUT_DIR,
        output_format="structured",
        verbose=False
    )


SoundCloud ASR Transcription Pipeline
Profile: https://soundcloud.com/radio-ergo
Date Range: 2024-07-01 to 2024-07-01
Output: /teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/02_intermediate/transcripts/mustafaa4a_ASR-Somali/transcriptions_database.csv



Processing tracks:   0%|          | 0/1 [00:00<?, ?track/s, 2024-07-01]

Processing tracks: 100%|██████████| 1/1 [02:11<00:00, 131.06s/track, 2024-07-01]


Processing Complete
✓ Successful: 1
✗ Failed: 0
Database: /teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/02_intermediate/transcripts/mustafaa4a_ASR-Somali/transcriptions_database.csv



In [6]:
import pandas as pd

def load_transcription_data(file_path: str) -> pd.DataFrame | None:
    """
    Loads transcription data from a CSV file into a pandas DataFrame.

    Args:
        file_path: The path to the CSV file.

    Returns:
        A pandas DataFrame containing the data, or None if the file is not found.
    """
    try:
        # Read the CSV file into a DataFrame
        df = pd.read_csv(file_path)
        print("File loaded successfully!")
        return df
    except FileNotFoundError:
        print(f"Error: The file was not found at the path: {file_path}")
        print("Please make sure the file is uploaded and the path is correct.")
        return None

# --- Usage Example ---

# Define the name of your file
# This assumes the file is in the same directory as your script
file_name = '/teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/02_intermediate/transcripts/mustafaa4a_ASR-Somali/transcriptions_database.csv'

# Load the data using the function
transcriptions_df = load_transcription_data(file_name)

# If the DataFrame was loaded successfully, display the first 5 rows
if transcriptions_df is not None:
    print("\nHere's a preview of your data:")
    print(transcriptions_df.head())

File loaded successfully!

Here's a preview of your data:
                                 id  \
0  d2fb77e8a8ab60bc5efc69d0cc721324   

                                                 url                  title  \
0  https://soundcloud.com/radio-ergo/idaacadda-01...  IDAACADDA 01-JUL-2024   

  source_type  date_recorded              date_processed transcription_method  \
0  soundcloud       20240701  2025-10-07T14:22:32.784936          huggingface   

              model_used  processing_duration_seconds  audio_size_mb  \
0  Mustafaa4a/ASR-Somali                    42.861944      54.782086   

   audio_duration_seconds audio_format  transcript_length_chars  \
0                3590.139          mp3                    50680   

   transcript_length_words language_detected processing_status  error_message  \
0                     7524            somali           success            NaN   

   file_path                                    transcript_text  
0        NaN  halkani waa raadya

In [7]:
transcriptions_df

,id,url,title,source_type,date_recorded,date_processed,transcription_method,model_used,processing_duration_seconds,audio_size_mb,audio_duration_seconds,audio_format,transcript_length_chars,transcript_length_words,language_detected,processing_status,error_message,file_path,transcript_text
0,d2fb77e8a8ab60bc5efc69d0cc721324,https://soundcloud.com/radio-ergo/idaacadda-01...,IDAACADDA 01-JUL-2024,soundcloud,20240701,2025-10-07T14:22:32.784936,huggingface,Mustafaa4a/ASR-Somali,42.861944,54.782086,3590.139,mp3,50680,7524,somali,success,NaN,NaN,halkani waa raadyahay argoee codka arrimaha ba...


In [ ]:
ok great so now i want to prepare this code for bigger amounts of data which i want to download so probably i will be doing in parts so i think i should have some mechanism which will check what was already downloaded what do you think, because i would want to download data from 2020-01-01 to 2025-09-30 so it will be massive because one 